# Modelos Fundacionales de Visión y Lenguaje

## Contrastive Language-Image Pre-Training (CLIP)

CLIP es una red neuronal entrenada en un conjunto de pares de imagenes y texto. Este modelo está entrenado para escoger la descripción de texto más relevante dada una imagen. Adicionalmente, tiene capacidades de generalización similares a las de GPT-2 y 3. CLIP iguala el rendimiento de ResNet50 en ImageNet en un escenario "zero-shot" sin utilizar ninguno de los 1,28 millones de ejemplos etiquetados en el conjunto de entrenamiento de esta base de datos, superando varios desafíos importantes en la visión por computador.

Durante este tutorial nos enfocaremos en el funcionamiento de esta arquitectura. Para esto, nos basaremos en la implementación abierta de la librería de openclip. A partir de esto, evaluaremos sus capacidades de reconocimiento para el dataset de Caltecth-101.

## Preparación para Colab

Asegúrese de estar ejecutando un entorno de ejecución de GPU; de lo contrario, seleccione "GPU" como acelerador de hardware en Entorno de ejecución > Cambiar tipo de entorno de ejecución en el menú. Las próximas celdas instalarán el paquete `openclip` y sus dependencias, y comprobarán si PyTorch 1.7.1 o posterior está instalado.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# Importamos las librerías requeridas
import numpy as np
import torch
import open_clip
from open_clip import tokenizer
from safetensors.torch import load_model


## Cargar el modelo

`open_clip.list_pretrained()` lista los nombres de los modelos de CLIP disponibles.

In [ ]:
open_clip.list_pretrained()

In [ ]:

# Cargamos el modelo "convnext_base_w" y los pesos preentrenados en el dataset "laion2b_s13b_b82k_augreg"
model, _, preprocess = open_clip.create_model_and_transforms('convnext_base_w')
load_model(model, 'laion2b_s13b_b82k_augreg.safetensors')
model.eval()
context_length = model.context_length
vocab_size = model.vocab_size

print("Parámetros del modelo:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Longitud de contexto:", context_length)
print("Tamaño del vocabulario:", vocab_size)

## Preprocesamiento de las imágenes

Como preprocesamiento, se normaliza la intensidad de los píxeles utilizando la media y la desviación estándar del conjunto de datos, aplicamos una operación de reescalamiento y un recorte en el centro para ajustar las imágenes de entrada a la resolución de imagen que espera el modelo.
El segundo valor de retorno de `open_clip.create_model_and_transforms()` contiene un objeto de tipo torchvision Transform con las operacione necesarias para este preprocesamiento.

In [ ]:
preprocess

## Preprocesamiento del texto

Utilizamos un tokenizador que no distingue entre mayúsculas y minúsculas, que se puede ejecutar usando la función `tokenizer.tokenize()`. De forma predeterminada, las salidas se rellenan para alcanzar una longitud de 77 tokens, que es el número de token que esperan los modelos CLIP.

In [ ]:
from open_clip import tokenizer
tokenizer.tokenize("Hello World!")

## Configuración de imágenes y textos de entrada

Vamos a introducir 8 imágenes de ejemplo y sus descripciones textuales en el modelo y compararemos la similitud entre las representaciones correspondientes.
El tokenizador no distingue entre mayúsculas y minúsculas y podemos proporcionar libremente cualquier descripción textual que se ajuste a nuestras necesidades.

In [ ]:
import os
import skimage
import IPython.display
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

from collections import OrderedDict
import torch

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# images in skimage to use and their textual descriptions
descriptions = {
    "page": "a page of text about segmentation",
    "chelsea": "a facial photo of a tabby cat",
    "astronaut": "a portrait of an astronaut with the American flag",
    "rocket": "a rocket standing on a launchpad",
    "stereo_motorcycle": "a red motorcycle standing in a garage",
    "camera": "a person looking at a camera on a tripod",
    "horse": "a black-and-white silhouette of a horse",
    "coffee": "a cup of coffee on a saucer"
}

In [ ]:
# Visualizamos las imágenes de ejemplo junto con sus descripciones
original_images = []
images = []
texts = []
plt.figure(figsize=(16, 5))

for name in descriptions:
    image = getattr(skimage.data, name)
    image = image()
    if name == "stereo_motorcycle":
      image = image[0]

    plt.subplot(2, 4, len(images) + 1)
    plt.imshow(image, cmap="gray")
    plt.title(f"{name}\n{descriptions[name]}")
    plt.xticks([])
    plt.yticks([])

    original_images.append(image)
    image = Image.fromarray(image)
    images.append(preprocess(image))
    texts.append(descriptions[name])

plt.tight_layout()

## Cálculo de las representaciones

Calculamos las representaciones de las imágenes y las descripciones de texto utilizando los encoder de visión y lenguaje, respectivamente.

In [ ]:
image_input = torch.tensor(np.stack(images))
text_tokens = tokenizer.tokenize(["This is " + desc for desc in texts]) # Tokenizamos las descripciones de texto
with torch.no_grad():
    image_features = model.encode_image(image_input).float() # Calculamos las representaciones de las imágenes
    text_features = model.encode_text(text_tokens).float() # Calculamos las representaciones de las descripciones de texto

## Cálculo de la similitud del coseno

Normalizamos las características y calculamos el producto escalar de cada par.

In [ ]:
# Normalizamos las representaciones de las imágenes y las descripciones de texto
image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)
# Calculamos la similaridad de coseno a partir del producto escalar de cada par.
similarity = text_features.cpu().numpy() @ image_features.cpu().numpy().T

In [ ]:
# Visualizamos los resultados
count = len(descriptions)

plt.figure(figsize=(20, 14))
plt.imshow(similarity, vmin=0.1, vmax=0.3)
plt.yticks(range(count), texts, fontsize=18)
plt.xticks([])
for i, image in enumerate(original_images):
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower", cmap="gray")
for x in range(similarity.shape[1]):
    for y in range(similarity.shape[0]):
        plt.text(x, y, f"{similarity[y, x]:.2f}", ha="center", va="center", size=12)

for side in ["left", "top", "right", "bottom"]:
  plt.gca().spines[side].set_visible(False)

plt.xlim([-0.5, count - 0.5])
plt.ylim([count + 0.5, -2])

plt.title("Cosine similarity between text and image features", size=20)

## Clasificación de imágenes "Zero-shot"

Puede clasificar imágenes utilizando la similitud del coseno (multiplicada por 100) como logits para la operación softmax.

In [ ]:
# Descargamos la base de datos CIFAR-100 para clasificars nuestras imágenes de ejemplo usando las 100 clases posibles en esta base de datos.
from torchvision.datasets import CIFAR100

cifar100 = CIFAR100(os.path.expanduser("~/.cache"), transform=preprocess, download=True)

In [ ]:
text_descriptions = [f"A photo of a {label}" for label in cifar100.classes] # Creamos una descripción de texto para cada categoría usando el template "A photo of a <category>"
text_tokens = tokenizer.tokenize(text_descriptions) # Preprocesamos las descripciones de texto

In [ ]:
# Calculamos las representaciones de texto y las normalizamos
with torch.no_grad():
    text_features = model.encode_text(text_tokens).float()
    text_features /= text_features.norm(dim=-1, keepdim=True)

# Calculamos la similaridad de coseno (multiplicada por 100) entre las imágenes de ejemplo y las 100 categorías de CIFAR-100. Finalmente, aplicamos la operación de Softmax
text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)
# Seleccionamos las 5 categorías con mayor probabilidad para cada imagen
top_probs, top_labels = text_probs.cpu().topk(5, dim=-1)

In [ ]:
# Visualizamos los resultadpos obtenidos
plt.figure(figsize=(16, 16))

for i, image in enumerate(original_images):
    plt.subplot(4, 4, 2 * i + 1)
    plt.imshow(image, cmap="gray")
    plt.axis("off")

    plt.subplot(4, 4, 2 * i + 2)
    y = np.arange(top_probs.shape[-1])
    plt.grid()
    plt.barh(y, top_probs[i])
    plt.gca().invert_yaxis()
    plt.gca().set_axisbelow(True)
    plt.yticks(y, [cifar100.classes[index] for index in top_labels[i].numpy()])
    plt.xlabel("probability")

plt.subplots_adjust(wspace=0.5)
plt.show()